# Milestone 1

In [3]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats.mstats import winsorize
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.impute import SimpleImputer
from datetime import datetime
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelBinarizer
import shap
from itertools import product
import plotly.graph_objects as go
from minisom import MiniSom
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import OPTICS

In [4]:
data=pd.read_csv("data\\financial_loan.csv")

In [5]:
data = data.drop(['int_rate', 'grade', 'sub_grade'], axis=1)

In [6]:
data = data.drop(['id', 'member_id', 'application_type'], axis=1)

In [7]:
data = data.drop(['installment', 'total_payment'], axis=1) #XXX ważne do późniejszego testowania

In [8]:
data['is_employed'] = np.where(data['emp_title'].isnull(), 0, 1)
data = data.drop('emp_title', axis=1)

In [9]:
data['loan_amount'] = winsorize(data['loan_amount'], limits=[0, 0.04])
data['annual_income'] = np.log1p(data['annual_income'])
data['total_acc'] = winsorize(data['total_acc'], limits=[0, 0.02])

In [10]:
numeric_cols = data.select_dtypes(include=['int64', 'float64', 'int32']).columns
scaler = StandardScaler()
data[numeric_cols] = scaler.fit_transform(data[numeric_cols])

In [11]:
date_cols = ['issue_date', 'last_credit_pull_date', 'last_payment_date', 'next_payment_date']
for col in date_cols:
    data[col] = pd.to_datetime(data[col], errors='coerce', dayfirst=True)

data['issue_month'] = data['issue_date'].dt.month
data['issue_day'] = data['issue_date'].dt.day


In [12]:
dd = data['last_credit_pull_date']-data['issue_date']
dd_diff=[]
for i in dd:
    dd_diff.append(float(str(i).split(" ")[0]))
data['date_diff']=pd.Series(dd_diff) #różnica dni pomiędzy wydaniem kredytu a sprawdzeniem historii kredytowej

data['date_diff'] = data['date_diff'].astype(float)

data['date_diff'] = scaler.fit_transform(data[['date_diff']])


In [13]:
data = data.drop(['issue_date', 'last_credit_pull_date', 'last_payment_date', 'next_payment_date'], axis=1)

In [14]:
emp_length_order = ['< 1 year', '1 year', '2 years', '3 years', '4 years', 
                    '5 years', '6 years', '7 years', '8 years', '9 years', '10+ years']

ord_encoder = OrdinalEncoder(categories=[emp_length_order])
data[['emp_length']] = ord_encoder.fit_transform(data[['emp_length']])

In [15]:
slownik = {
    'położenie': {
        'Northeast': ['CT', 'ME', 'MA', 'NH', 'RI', 'VT', 'NJ', 'NY', 'PA'],
        'Midwest': ['IL', 'IN', 'MI', 'OH', 'WI', 'IA', 'KS', 'MN', 'MO', 'NE', 'ND', 'SD'],
        'South': ['DE', 'DC', 'FL', 'GA', 'MD', 'NC', 'SC', 'VA', 'WV', 'AL', 'KY', 'MS', 'TN', 'AR', 'LA', 'OK', 'TX'],
        'West': ['AZ', 'CO', 'ID', 'MT', 'NV', 'NM', 'UT', 'WY', 'AK', 'CA', 'HI', 'OR', 'WA']
    },
    'zamożność': {
        'high': ['DC', 'MD', 'MA', 'CA', 'CO', 'NJ', 'WA', 'AK', 'VA', 'NH', 'UT', 'HI', 'CT', 'MN', 'NY', 'RI', 'DE', 'NV', 'IL', 'TX', 'AZ', 'GA', 'WI', 'OR', 'ND', 'VT', 'FL', 'NE', 'NC', 'PA', 'SD', 'IA', 'MI', 'IN', 'OH', 'SC', 'KS', 'MO', 'TN', 'ME'],
        'med': ['NM', 'ID', 'LA', 'OK', 'MT', 'KY', 'AL', 'WV', 'AR', 'MS'],
        'low': ['WY', 'NM', 'MS', 'WV', 'AR', 'AL', 'KY', 'OK', 'LA', 'SC']
    },
    'prawo_kredytowe': {
        'restrictive': ['PA', 'NY', 'CA', 'AZ', 'HI', 'IN', 'IA', 'KS', 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'ND', 'OH', 'OK', 'OR', 'SD', 'TN', 'UT', 'WV', 'WI', 'WY'],
        'moderate': ['DE', 'FL', 'GA', 'ID', 'IL', 'KY', 'ME', 'MD', 'MA', 'MI', 'MN', 'MO', 'NC', 'RI', 'TX', 'VT', 'VA', 'WA'],
        'lenient': ['AK', 'AL', 'AR', 'CO', 'CT', 'DC', 'LA', 'MS', 'SC', 'WI', 'WY', 'SD']
    }
}


In [16]:
# Tworzymy odwrócone słowniki dla każdej kategorii
położenie_map = {state: region for region, states in slownik['położenie'].items() for state in states}
zamożność_map = {state: zamożność for zamożność, states in slownik['zamożność'].items() for state in states}
prawo_map = {state: prawo for prawo, states in slownik['prawo_kredytowe'].items() for state in states}

# Mapowanie kolumny address_state
data['location'] = data['address_state'].map(położenie_map)
data['wealth'] = data['address_state'].map(zamożność_map)
data['law_regulations'] = data['address_state'].map(prawo_map)

In [17]:
data = pd.get_dummies(data, columns=['location', 'wealth', 'law_regulations'], drop_first=True)
data = data.drop('address_state', axis=1)

In [18]:
top_purpose = data['purpose'].value_counts().nlargest(2).index
data['purpose_grouped'] = data['purpose'].apply(lambda x: x if x in top_purpose else 'other')
data = pd.get_dummies(data, columns=['purpose_grouped'], drop_first=True)

data = data.drop('purpose', axis=1)

In [19]:
cat_cols = ['verification_status', 'loan_status', 'home_ownership']
data = pd.get_dummies(data, columns=cat_cols, drop_first=True)

bool_cols = data.select_dtypes(include='bool').columns
data[bool_cols] = data[bool_cols].astype(int)

In [20]:
data['term'] = data['term'].astype(str).str.strip()
data['60_month_term'] = np.where(data['term'] == '60 months', 1, 0)
data = data.drop('term', axis=1)


In [21]:
data['issue_month_sin'] = np.sin(2 * np.pi * data['issue_month'] / 12)
data['issue_month_cos'] = np.cos(2 * np.pi * data['issue_month'] / 12)
data['issue_day_sin'] = np.sin(2 * np.pi * data['issue_day'] / 31)
data['issue_day_cos'] = np.cos(2 * np.pi * data['issue_day'] / 31)
cols_to_scale = ['date_diff', 'emp_length', 
                 'issue_month_sin', 'issue_month_cos', 
                 'issue_day_sin', 'issue_day_cos']

scaler = StandardScaler()
data[cols_to_scale] = scaler.fit_transform(data[cols_to_scale])
data = data.drop(['issue_month', 'issue_day'], axis=1)



In [22]:
X = data.select_dtypes(include=[float, int])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)


In [23]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X)

In [24]:
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = umap_model.fit_transform(X)

c:\Users\czare\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\czare\AppData\Local\Programs\Python\Python311\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [25]:
pca = PCA(n_components=X.shape[1], random_state=42)
pca.fit(X)

PCA(n_components=29, random_state=42)

In [26]:
def model_eval(X, labels):
    unique_labels = set(labels)
    if len(unique_labels) < 2 or (len(unique_labels) == 1 and -1 in unique_labels):
        return {'silhouette': None, 'calinski': None, 'davies': None}
    
    silhouette = silhouette_score(X, labels)
    calinski = calinski_harabasz_score(X, labels)
    davies = davies_bouldin_score(X, labels)
    
    print(f'silhouette: {silhouette:.3f}, calinski: {calinski:.3f}, davies: {davies:.3f}')
    
    return {'silhouette': silhouette, 'calinski': calinski, 'davies': davies}

In [27]:
def kmeans_grid_search(X,
                       init_list=["k-means++", "random"],
                       n_init_list=[10, 20],
                       algo_list=["lloyd", "elkan"],
                       max_iter_list=[300],
                       k_list=range(2, 8),
                       scoring="silhouette"):

    results = []

    for init, n_init, algo, max_iter, k in product(init_list, n_init_list, algo_list, max_iter_list, k_list):
        try:
            model = KMeans(n_clusters=k, init=init, n_init=n_init,
                           algorithm=algo, max_iter=max_iter, random_state=42)
            labels = model.fit_predict(X)
            score = silhouette_score(X, labels) if scoring == "silhouette" else None
            results.append({
                "model": "KMeans",
                "params": f"k={k}, init={init}, n_init={n_init}, algo={algo}",
                "score": score
            })
        except Exception:
            continue

    return pd.DataFrame(results).sort_values(by="score", ascending=False)


def agglo_grid_search(X, linkage_list=["ward", "complete", "average"], k_list=range(2, 8), scoring="silhouette"):
    results = []

    for linkage, k in product(linkage_list, k_list):
        try:
            model = AgglomerativeClustering(n_clusters=k, linkage=linkage)
            labels = model.fit_predict(X)
            score = silhouette_score(X, labels) if scoring == "silhouette" else None
            results.append({
                "model": "Agglomerative",
                "params": f"k={k}, linkage={linkage}",
                "score": score
            })
        except Exception:
            continue

    return pd.DataFrame(results).sort_values(by="score", ascending=False)


def gmm_grid_search(X, covariance_list=["full", "tied", "diag", "spherical"], k_list=range(2, 8), scoring="silhouette"):
    results = []

    for cov_type, k in product(covariance_list, k_list):
        try:
            model = GaussianMixture(n_components=k, covariance_type=cov_type, random_state=42)
            labels = model.fit(X).predict(X)
            score = silhouette_score(X, labels) if scoring == "silhouette" else None
            results.append({
                "model": "GMM",
                "params": f"k={k}, covariance={cov_type}",
                "score": score
            })
        except Exception:
            continue

    return pd.DataFrame(results).sort_values(by="score", ascending=False)


def dbscan_grid_search(X, eps_list=[0.3, 0.5, 0.7, 1.0], min_samples_list=[3, 5, 10], scoring="silhouette"):
    results = []

    for eps, min_s in product(eps_list, min_samples_list):
        try:
            model = DBSCAN(eps=eps, min_samples=min_s)
            labels = model.fit_predict(X)
            if len(set(labels)) <= 1 or len(set(labels)) >= len(X):  # skip noise-only or all-unique
                score = None
            else:
                score = silhouette_score(X, labels) if scoring == "silhouette" else None
            results.append({
                "model": "DBSCAN",
                "params": f"eps={eps}, min_samples={min_s}",
                "score": score
            })
        except Exception:
            continue

    return pd.DataFrame(results).sort_values(by="score", ascending=False)


In [28]:
def find_optimal_clusters_multi_algorithm(X, algorithms=None, max_k=20):
    """
    Evaluate multiple clustering algorithms across different k values
    """
    
    if algorithms is None:
        algorithms = {
            'KMeans': KMeans,
            'AgglomerativeClustering': AgglomerativeClustering,
            'GaussianMixture': GaussianMixture
        }
    
    results = {}
    k_range = range(2, max_k + 1)
    
    print("Multi-Algorithm Cluster Evaluation:")
    print("=" * 60)
    
    for algo_name, algo_class in algorithms.items():
        print(f"\n{algo_name} Results:")
        print("-" * 40)
        
        silhouette_scores = []
        calinski_scores = []
        davies_scores = []
        inertias = []
        
        for k in k_range:
            print(f"\nK = {k}:")
            
            try:
                # Handle different algorithm parameter names
                if algo_name == 'GaussianMixture':
                    model = algo_class(n_components=k, random_state=42)
                    model.fit(X)
                    labels = model.predict(X)
                    inertia = None  # GMM doesn't have inertia
                elif algo_name == 'AgglomerativeClustering':
                    model = algo_class(n_clusters=k)
                    labels = model.fit_predict(X)
                    inertia = getattr(model, 'inertia_', None)
                else:
                    model = algo_class(n_clusters=k, random_state=42)
                    labels = model.fit_predict(X)
                    inertia = getattr(model, 'inertia_', None)
                
                # Evaluate clustering
                metrics = model_eval(X, labels)
                
                silhouette_scores.append(metrics['silhouette'])
                calinski_scores.append(metrics['calinski'])
                davies_scores.append(metrics['davies'])
                inertias.append(inertia)
                
            except Exception as e:
                print(f"Error with {algo_name} at k={k}: {e}")
                silhouette_scores.append(None)
                calinski_scores.append(None)
                davies_scores.append(None)
                inertias.append(None)
        
        # Store results
        results[algo_name] = {
            'silhouette_scores': silhouette_scores,
            'calinski_scores': calinski_scores,
            'davies_scores': davies_scores,
            'inertias': inertias
        }
    
    # Plot comparison
    plot_algorithm_comparison(results, k_range)
    
    # Find best k for each algorithm and metric
    print_best_k_summary(results, k_range)
    
    return results

def plot_algorithm_comparison(results, k_range):
    """Plot comparison of different algorithms across metrics"""
    
    algorithms = list(results.keys())
    n_algorithms = len(algorithms)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    metrics = ['silhouette_scores', 'calinski_scores', 'davies_scores', 'inertias']
    titles = ['Silhouette Score (Higher is Better)', 
              'Calinski-Harabasz Score (Higher is Better)',
              'Davies-Bouldin Score (Lower is Better)', 
              'Inertia/WCSS (Lower is Better)']
    
    colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown']
    
    for i, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[i]
        
        for j, algo in enumerate(algorithms):
            scores = results[algo][metric]
            # Filter out None values
            valid_k = [k for k, score in zip(k_range, scores) if score is not None]
            valid_scores = [score for score in scores if score is not None]
            
            if valid_scores:  # Only plot if we have valid scores
                ax.plot(valid_k, valid_scores, marker='o', 
                       label=algo, color=colors[j % len(colors)])
        
        ax.set_title(title)
        ax.set_xlabel('Number of clusters (k)')
        ax.set_ylabel(title.split('(')[0].strip())
        ax.legend()
        ax.grid(True)
    
    plt.tight_layout()
    plt.show()

def print_best_k_summary(results, k_range):
    """Print summary of best k for each algorithm and metric"""
    
    print(f"\n{'='*60}")
    print("BEST K SUMMARY")
    print(f"{'='*60}")
    
    for algo in results.keys():
        print(f"\n{algo}:")
        
        # Silhouette (higher is better)
        sil_scores = results[algo]['silhouette_scores']
        valid_sil = [(k, score) for k, score in zip(k_range, sil_scores) if score is not None]
        if valid_sil:
            best_k_sil = max(valid_sil, key=lambda x: x[1])
            print(f"  Best k (Silhouette): {best_k_sil[0]} (score: {best_k_sil[1]:.3f})")
        
        # Calinski-Harabasz (higher is better)
        cal_scores = results[algo]['calinski_scores']
        valid_cal = [(k, score) for k, score in zip(k_range, cal_scores) if score is not None]
        if valid_cal:
            best_k_cal = max(valid_cal, key=lambda x: x[1])
            print(f"  Best k (Calinski-Harabasz): {best_k_cal[0]} (score: {best_k_cal[1]:.3f})")
        
        # Davies-Bouldin (lower is better)
        dav_scores = results[algo]['davies_scores']
        valid_dav = [(k, score) for k, score in zip(k_range, dav_scores) if score is not None]
        if valid_dav:
            best_k_dav = min(valid_dav, key=lambda x: x[1])
            print(f"  Best k (Davies-Bouldin): {best_k_dav[0]} (score: {best_k_dav[1]:.3f})")


In [29]:
X_pca = PCA(n_components=20, random_state=42).fit_transform(X)

In [30]:
#results_kmeans = kmeans_grid_search(X_pca)
#print(results_kmeans)

In [31]:
#results_agglo = agglo_grid_search(X_pca)
#print(results_agglo)

In [32]:
#results_gmm = gmm_grid_search(X_pca)
#print(results_gmm)

In [33]:
#results_dbscan = dbscan_grid_search(X_pca)
#print(results_dbscan)

In [34]:
# X_pca_40 = PCA(n_components=40, random_state=42).fit_transform(X)

In [35]:
# results_kmeans_40 = kmeans_grid_search(X_pca_40)
# print(results_kmeans_40)

In [36]:
# results_agglo_40 = agglo_grid_search(X_pca_40)
# print(results_agglo_40)

In [37]:
# results_gmm_40 = gmm_grid_search(X_pca_40)
# print(results_gmm_40)

In [38]:
# results_dbscan_40 = dbscan_grid_search(X_pca_40)
# print(results_dbscan_40)

In [39]:
# algorithms = {
#             'KMeans': KMeans,
#             'AgglomerativeClustering': AgglomerativeClustering,
#             'GaussianMixture': GaussianMixture
#         }
        
# results = find_optimal_clusters_multi_algorithm(X_pca, max_k=20, algorithms=algorithms)

In [40]:
def plot_tsne_umap_clusters(X_tsne, X_umap, labels, title_tsne='t-SNE', title_umap='UMAP'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # t-SNE
    sc1 = axes[0].scatter(X_tsne[:,0], X_tsne[:,1], c=labels, s=5, alpha=0.6, cmap='Spectral')
    axes[0].set_title(title_tsne)
    plt.colorbar(sc1, ax=axes[0], label='Cluster Label')
    
    # UMAP
    sc2 = axes[1].scatter(X_umap[:,0], X_umap[:,1], c=labels, s=5, alpha=0.6, cmap='Spectral')
    axes[1].set_title(title_umap)
    plt.colorbar(sc2, ax=axes[1], label='Cluster Label')
    
    plt.tight_layout()
    plt.show()


In [41]:
def plot_3d_clusters(X_tsne_3d, labels, title='3D Cluster Visualization'):
    fig = go.Figure(data=[go.Scatter3d(
        x=X_tsne_3d[:, 0],
        y=X_tsne_3d[:, 1],
        z=X_tsne_3d[:, 2],
        mode='markers',
        marker=dict(
            size=5,
            color=labels,
            colorscale='Spectral',
            opacity=0.7,
            colorbar=dict(title='Cluster Label')
        ),
        text=[f'Cluster: {lbl}' for lbl in labels]
    )])
    fig.update_layout(
    scene_camera_eye=dict(x=0.8, y=0.8, z=0.8) 
    )

    fig.show()


In [42]:
labels_dict = {}

In [ ]:
kmeans = KMeans(n_clusters=4, init = 'k-means++', n_init=20, algorithm= 'lloyd', random_state=42).fit(X)
labels_dict['kmeans1'] = kmeans.labels_

In [ ]:
# Agglomerative with average linkage
agglomerative1 = AgglomerativeClustering(n_clusters=4, linkage='average').fit(X)
labels_dict['agglomerative1'] = agglomerative1.labels_

# Agglomerative with ward linkage  
agglomerative2 = AgglomerativeClustering(n_clusters=4, linkage='ward').fit(X)
labels_dict['agglomerative2'] = agglomerative2.labels_

# Agglomerative with complete linkage
agglomerative3 = AgglomerativeClustering(n_clusters=4, linkage='complete').fit(X)
labels_dict['agglomerative3'] = agglomerative3.labels_

# Agglomerative with default linkage
agglomerative4 = AgglomerativeClustering(n_clusters=4).fit(X)
labels_dict['agglomerative4'] = agglomerative4.labels_


In [ ]:
# Gaussian with full covariance
gaussian1 = GaussianMixture(n_components=4, covariance_type='full', random_state=42).fit(X)
labels_dict['gaussian1'] = gaussian1.predict(X)

# Gaussian with spherical covariance
gaussian2 = GaussianMixture(n_components=4, covariance_type='spherical', random_state=42).fit(X)
labels_dict['gaussian2'] = gaussian2.predict(X)

# Gaussian with tied covariance
gaussian3 = GaussianMixture(n_components=4, covariance_type='tied', random_state=42).fit(X)
labels_dict['gaussian3'] = gaussian3.predict(X)

# Gaussian with diagonal covariance
gaussian4 = GaussianMixture(n_components=4, covariance_type='diag', random_state=42).fit(X)
labels_dict['gaussian4'] = gaussian4.predict(X)


In [ ]:
# OPTICS clustering
optics1 = OPTICS(min_samples=5, xi=0.05, min_cluster_size=0.1).fit(X)
labels_dict['optics1'] = optics1.labels_

# DBSCAN with eps=0.3
dbscan1 = DBSCAN(eps=0.3, min_samples=5).fit(X)
labels_dict['dbscan1'] = dbscan1.labels_

# DBSCAN with eps=1.0
dbscan2 = DBSCAN(eps=1.0, min_samples=10).fit(X)
labels_dict['dbscan2'] = dbscan2.labels_


In [47]:
# Załóżmy, że X_pca to Twoje dane po PCA, np. o wymiarach (n_samples, n_components)
# np. X_pca = pca.fit_transform(X)

# Parametry SOM
som_size = 7  # rozmiar siatki SOM (np. 7x7)
input_len = X_pca.shape[1]  # liczba cech po PCA

# Inicjalizacja SOM
som = MiniSom(x=4, y=1, input_len=input_len, sigma=0.5, learning_rate=0.5, neighborhood_function='gaussian', random_seed=42)

# Trening SOM (np. 1000 iteracji)
som.train_batch(X_pca, 100000, verbose=True)

# Przypisanie każdego punktu do neuronu (klastra)
winner_coordinates = np.array([som.winner(x) for x in X_pca])  # współrzędne zwycięskich neuronów
cluster_index = np.ravel_multi_index(winner_coordinates.T, (som_size, som_size))  # spłaszczone indeksy klastrów


 [ 100000 / 100000 ] 100% - 0:00:00 left 
 quantization error: 3.9947031856318245


In [48]:

# t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_pca)

# UMAP
umap_model = umap.UMAP(n_components=2, random_state=42)
X_umap = umap_model.fit_transform(X_pca)

c:\Users\czare\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\czare\AppData\Local\Programs\Python\Python311\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


# Interpretowalność

In [ ]:
def interpret_clustering_with_labels(X, labels_dict, random_state=42):
    """
    Evaluates interpretability of clustering solutions using SHAP feature importances,
    given precomputed cluster labels.
    
    Parameters:
        X (array-like): Feature matrix (n_samples, n_features).
        labels_dict (dict): Dictionary of label arrays, e.g., {'KMeans': labels1, ...}
        random_state (int): Random state for reproducibility.
    
    Returns:
        dict: For each label set, a dict with 'shap_values' and 'feature_importance' arrays.
    """
    results = {}
    
    for name, labels in labels_dict.items():
        # Encode labels if not numeric
        labels = LabelEncoder().fit_transform(labels)
        
        # Train classifier to predict cluster labels from features
        clf = RandomForestClassifier(random_state=random_state)
        clf.fit(X, labels)
        
        # SHAP analysis
        explainer = shap.TreeExplainer(clf)
        shap_values = explainer.shap_values(X)
        
        # Aggregate feature importances (mean absolute SHAP values)
        if isinstance(shap_values, list):  # Multiclass
            feature_importance = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
        else:
            feature_importance = np.abs(shap_values).mean(axis=0)
        
        results[name] = {
            'shap_values': shap_values,
            'feature_importance': feature_importance
        }
    
    return results

In [ ]:
X_sample = X.sample(n=5000, random_state=42)

In [ ]:
labels_dict['kmeans1'] = kmeans.labels_

labels_dict['agglomerative1'] = agglomerative1.labels_

labels_dict['agglomerative2'] = agglomerative2.labels_

labels_dict['agglomerative3'] = agglomerative3.labels_

labels_dict['agglomerative4'] = agglomerative4.labels_

labels_dict['gaussian1'] = gaussian1.predict(X)

labels_dict['gaussian2'] = gaussian2.predict(X)

labels_dict['gaussian3'] = gaussian3.predict(X)

labels_dict['gaussian4'] = gaussian4.predict(X)

labels_dict['optics1'] = optics1.labels_

labels_dict['dbscan1'] = dbscan1.labels_

labels_dict['dbscan2'] = dbscan2.labels_


In [ ]:
interpret_clustering_with_labels(X_sample, labels_dict, random_state=42)

ValueError: Found input variables with inconsistent numbers of samples: [5000, 38576]